In [1]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_PASSWORD",
    "database": "csi3"
}

CSV_FILE = r"C:\Users\pranj\Downloads\archive (3)\Sample - Superstore.csv"
TABLE_NAME = "superstore_raw"

try:
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()

    df = pd.read_csv(CSV_FILE, encoding="latin1")

    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace(".", "", regex=False)
    )

    cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")

    column_definitions = []

    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            datatype = "BIGINT"
        elif pd.api.types.is_float_dtype(df[col]):
            datatype = "DOUBLE"
        else:
            datatype = "TEXT"

        column_definitions.append(f"`{col}` {datatype}")

    create_table_query = f"""
    CREATE TABLE {TABLE_NAME} (
        {', '.join(column_definitions)}
    )
    """

    cursor.execute(create_table_query)

    columns = ", ".join(f"`{col}`" for col in df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))

    insert_query = f"""
    INSERT INTO {TABLE_NAME} ({columns})
    VALUES ({placeholders})
    """

    data = [
        tuple(None if pd.isna(value) else value for value in row)
        for row in df.values
    ]

    cursor.executemany(insert_query, data)
    conn.commit()

    cursor.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
    row_count = cursor.fetchone()[0]

    print("Data imported successfully.")
    print(f"Database : csi3")
    print(f"Table    : {TABLE_NAME}")
    print(f"Rows     : {row_count}")

except Error as error:
    print(f"MySQL Error: {error}")

except Exception as error:
    print(f"Error: {error}")

finally:
    if 'cursor' in locals():
        cursor.close()

    if 'conn' in locals() and conn.is_connected():
        conn.close()

Data imported successfully.
Database : csi3
Table    : superstore_raw
Rows     : 9994
